# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). All datasets, record sets, fields, and columns are referenced by their `@id` as required by the Croissant specification.

In [ ]:
# Install required library
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print overview from metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {', '.join(meta.keywords or [])}")
print(f"License: {meta.license}")

## 2. Data Overview

Review available record sets, fields, and their `@id`. All references use the entity `@id` to ensure schema integrity.

Let's list the available record sets and, for each, the fields and column ids.

In [ ]:
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
overview = []
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    fields = rs.fields
    print(f"  Fields:")
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    overview.append({"record_set_id": rs.id, "fields": [(field.name, field.id, field.data_type) for field in fields]})
    print()

## 3. Data Extraction

Load data from record sets into DataFrames for analysis. Here, we focus on the main record set that holds protein records (determined by browsing the dataset schema overview above). Replace `<record_set_id>` below if you wish to analyze a different set.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
# For demonstration we pick the most relevant record set (likely  the main tabular data; usually first or named like 'protein_records')
main_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}")

if main_record_set_id and main_record_set_id in dataframes:
    print("\nColumns in primary record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Now let's process and explore the dataframe. We'll select one numeric field (referenced by its `@id` from the earlier data overview) for filtering and normalization, and one grouping field for aggregation.

In [ ]:
# You may update these to use other field @ids as needed
# For demonstration, pick known likely numeric and group fields from the dataset
numeric_candidate_ids = [col for col in dataframes[main_record_set_id].columns if 'abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower()]
group_candidate_ids = [col for col in dataframes[main_record_set_id].columns if 'description' in col.lower() or 'accession' in col.lower() or 'name' in col.lower() or 'sample' in col.lower()]

if numeric_candidate_ids:
    numeric_field = numeric_candidate_ids[0]
else:
    numeric_field = dataframes[main_record_set_id].select_dtypes(include=np.number).columns[0]
    # fallback

print(f"Numeric field selected (by @id): {numeric_field}")

# Set filter threshold and rename for clarity
threshold = dataframes[main_record_set_id][numeric_field].dropna().mean()  # e.g., above mean
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
if group_candidate_ids:
    group_field = group_candidate_ids[0]
    print(f"\nGrouping by {group_field} (@id)")
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_abundance').sort_values('mean_abundance', ascending=False)
    )
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field (e.g., protein abundance, peptide counts, or MW) and, if applicable, grouped means.


In [ ]:
# 1. Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
dataframes[main_record_set_id][numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# 2. Top 10 means by group, if grouping was performed
if 'grouped_df' in locals():
    grouped_df.head(10).plot(kind='bar', legend=None, figsize=(10, 4))
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion

- The dataset, referenced entirely by Croissant `@id`, was loaded and explored using `mlcroissant`.
- Key numeric and categorical fields were automatically detected by their likely schema roles (inspection of full record set, field IDs, and data values is recommended for further fine-tuning).
- Simple filtering, normalization, and group statistics provide insights into the data distribution and variable relationships.
- For biological datasets like this, further downstream analysis (e.g., annotation, enrichment, or benchmarking workflows) can be built on these foundations.

_For advanced analysis, consult the Croissant field descriptions and the FAIR^2 metadata to reference additional record sets or fields by `@id` under a reproducible, machine-actionable, and FAIR approach._